# 文档搜索中的可解释检索

## 概述

此代码实现了一个可解释的检索器，该系统不仅可以根据查询检索相关文档，还可以解释每个检索到的文档为何相关。它将基于向量的相似性搜索与自然语言解释相结合，增强了检索过程的透明度和可解释性。

## 动机

传统的文档检索系统通常作为黑匣子工作，提供结果而不解释为什么选择它们。在理解结果背后的推理至关重要的情况下，缺乏透明度可能会产生问题。可解释的检索器通过提供对每个检索到的文档的相关性的洞察来解决这个问题。

## 关键组件

1. 支持从输入文本创建存储
2. 使用FAISS进行高效相似性搜索的基础搜索器
3. 用于生成解释的语言模型（LLM）
4. 结合检索和解释生成的自定义Explainable搜索器类

## 方法详细信息

### 文档预处理和支持存储创建

1. 使用 OpenAI 的嵌入模型将输入文本转换为嵌入。
2. 根据这些嵌入创建 FAISS 存储存储，以实现高效的相似性搜索。

### 搜索器设置

1. 从支持存储创建一个基础搜索器，配置为返回前 5 个最相似的文档。

### 解释生成

1. LLM (GPT-4) 用于生成解释。
2. 定义自定义提示模板，指导LLM解释检索到的文档的相关性。

### 可解释的搜索器类

1. 将基本搜索器和解释生成合并到一个界面中。
2、`retrieve_and_explain`方法：
   - 使用基本搜索器检索相关文档。
   - 对于每个检索到的文档，生成其与查询相关性的解释。
   - 返回包含文档内容及其解释的字典列表。

## 这种方法的好处

1. 透明性：用户可以了解特定文档为何被检索。
2. 信任：解释可以增强用户对系统结果的信心。
3. 学习：用户可以深入了解查询和文档之间的关系。
4. 调试：更容易识别和纠正检索过程中的问题。
5. 定制：可以针对不同的用例或领域定制解释提示。

## 结论

可解释检索器代表了朝着更可解释和更值得信赖的信息检索系统迈出的重要一步。通过提供自然语言解释以及检索到的文档，它弥合了强大的基于矢量的搜索技术和人类理解之间的差距。这种方法在信息检索背后的推理与检索到的信息本身同样重要的各个领域都有潜在的应用，例如法律研究、医疗信息系统和教育工具。

# 包安装和导入

下面的单元格安装运行此笔记本所需的所有必需软件包。

In [ ]:
# Install required packages
!pip install python-dotenv

In [ ]:
# Clone the repository to access helper functions and evaluation modules
!git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git
import sys
sys.path.append('RAG_TECHNIQUES')
# If you need to run with the latest data
# !cp -r RAG_TECHNIQUES/data .

In [ ]:
import os
import sys
from dotenv import load_dotenv


# Original path append replaced for Colab compatibility
from helper_functions import *
from evaluation.evalute_rag import *

# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

### 定义可解释的搜索器类

In [6]:
class ExplainableRetriever:
    def __init__(self, texts):
        self.embeddings = OpenAIEmbeddings()

        self.vectorstore = FAISS.from_texts(texts, self.embeddings)
        self.llm = ChatOpenAI(temperature=0, model_name="gpt-4o-mini", max_tokens=4000)

        
        # Create a base retriever
        self.retriever = self.vectorstore.as_retriever(search_kwargs={"k": 5})
        
        # Create an explanation chain
        explain_prompt = PromptTemplate(
            input_variables=["query", "context"],
            template="""
            Analyze the relationship between the following query and the retrieved context.
            Explain why this context is relevant to the query and how it might help answer the query.
            
            Query: {query}
            
            Context: {context}
            
            Explanation:
            """
        )
        self.explain_chain = explain_prompt | self.llm

    def retrieve_and_explain(self, query):
        # Retrieve relevant documents
        docs = self.retriever.get_relevant_documents(query)
        
        explained_results = []
        
        for doc in docs:
            # Generate explanation
            input_data = {"query": query, "context": doc.page_content}
            explanation = self.explain_chain.invoke(input_data).content
            
            explained_results.append({
                "content": doc.page_content,
                "explanation": explanation
            })
        
        return explained_results



### 创建一个模拟示例和可解释的搜索器实例

In [7]:

# Usage
texts = [
    "The sky is blue because of the way sunlight interacts with the atmosphere.",
    "Photosynthesis is the process by which plants use sunlight to produce energy.",
    "Global warming is caused by the increase of greenhouse gases in Earth's atmosphere."
]

explainable_retriever = ExplainableRetriever(texts)


### 显示结果

In [ ]:
query = "Why is the sky blue?"
results = explainable_retriever.retrieve_and_explain(query)

for i, result in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"Content: {result['content']}")
    print(f"Explanation: {result['explanation']}")
    print()

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--explainable-retrieval)